# Random-address PRF: conditional per-stage CPA

This notebook corresponds to the **random-address experiments** in the accompanying
paper (Sections 5.2, 5.3 and 6.1–6.2). It evaluates each secret-word stage
with *correct preceding words*; a stage match **does not** represent successful
end-to-end SK.seed recovery. All experiments use cumulative-HW, low-byte-first
8-bit split CPA, beam width 4, SHA-256 updated A and the paper's message-schedule
routes. Real captured arrays and ground-truth manifests are needed locally.

The default public/demo PK.seed is **not** the original experiment's PK.seed.
For original measurements specify `PK_SEED_HEX` locally and never commit it.
The rank/GE figures of the paper are **not** generated by this notebook.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory")
sys.path.insert(0, str(REPO_ROOT / "src"))

import random_address_cpa as runner


## Configuration

Check the dataset path and keep `RUN_CPA = False` until ready.


In [ ]:
BASE_FOLDER = Path(os.environ.get("SLH_CPA_RANDOM_BASE", "./data/random_address"))
RESULTS_DIR = Path(os.environ.get("SLH_CPA_RANDOM_RESULTS", "./results/random_address"))
PARAM_SETS = ("128f", "128s", "192f", "192s", "256f", "256s")
COMPILER_OPTS = ("Os",)
ADDRESS_KINDS = ("fors", "wots")
SEED_IDS = tuple(range(30))          # Paper: 30 independent SK.seed values
MAX_TRACES = 5000                 # Paper: 5,000 per (parameter, kind, seed)
PK_SEED_HEX = os.environ.get("SLH_CPA_PK_SEED_HEX", "")  # one security level only if set
BACKEND = "auto"                 # "numpy" works without TensorFlow
RUN_CPA = False                  # Change only after preflight and reviewing inputs


## Data inventory and optional CPA execution

Dry runs inspect local measured datasets; they do not generate new measurements.


In [ ]:
def argv(output):
    args = ["--base", str(BASE_FOLDER), "--output", str(output),
            "--params", ",".join(PARAM_SETS), "--opts", ",".join(COMPILER_OPTS),
            "--kinds", ",".join(ADDRESS_KINDS),
            "--seed-ids", ",".join(map(str, SEED_IDS)),
            "--max-traces", str(MAX_TRACES), "--backend", BACKEND,
            "--attacks", "both", "--beam", "4", "--joint-classes", "4"]
    if PK_SEED_HEX:
        args += ["--pk-seed-hex", PK_SEED_HEX]
    return args

if not BASE_FOLDER.is_dir():
    print("Missing local capture folder:", BASE_FOLDER)
else:
    print("Preflight:", runner.main(argv(RESULTS_DIR / "_preview") + ["--dry-run"]))
    if RUN_CPA:
        print("CPA:", runner.main(argv(RESULTS_DIR)))
    else:
        print("Preview only. Set RUN_CPA=True after confirming capture/seed settings.")
